# 第3日 ノートブック01：YOLO を理解する

# Day 3 Notebook 01: Understanding YOLO

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. 物体検出モデルが何をするのか説明できる | 1. Explain what an object detection model does |
| 2. バウンディングボックスと座標の表し方を説明できる | 2. Explain bounding boxes and their coordinates |
| 3. 確からしさ（confidence）の意味を説明できる | 3. Explain what confidence means |
| 4. IoU を手計算できる | 4. Compute IoU by hand |
| 5. 事前学習モデルが PVC 欠陥を検出できない理由を説明できる | 5. Explain why a pretrained model cannot detect PVC defects |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 80 分 / 80 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第1日・第2日を終えていること。 / You have finished Days 1 and 2. |
| 必要なライブラリ / Required libraries | numpy, matplotlib, pillow |
| 使用カーネル / Jupyter kernel | aivision |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第3日 / Day 3](../../COURSE_SYLLABUS.md#day3) |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| IoU の手計算結果 | Your hand-computed IoU |
| YOLO 形式の座標計算 | A YOLO-format coordinate calculation |

### confidence と正しさは別 / confidence is not correctness

![](../../assets/diagrams/teach_score_vs_correctness.png)

図：5 つの検出の confidence と、正解と比べた正しさ（値は例示用）。

Figure: confidence against correctness for five detections (illustrative values).

**気づいてほしいこと**

confidence が高い検出でも間違っていることがあり、低い検出でも正しいことがあります。confidence はモデル自身の自信であって、正解ではありません。

**What to notice**

A high-confidence detection can be wrong and a low-confidence one can be right. Confidence is the model's own certainty, not ground truth.

**考えてみてください / Try to answer**

- しきい値 0.60 で残る検出のうち、正しくないのはどれですか。
- At a threshold of 0.60, which of the surviving detections are wrong?


## このノートブックの位置 / Where this notebook sits

| 項目 / Item | 内容 / Details |
| --- | --- |
| 日 / Day | 第3日 / Day 3 |
| この日の順番 / Position today | 1 / 3（この日のノートブックのうち）/ of today's notebooks |
| 所要時間 / Duration | 80 分 / 80 minutes |
| 前提 / Prerequisites | 第2日 / Day 2 |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第3日 / Day 3](../../COURSE_SYLLABUS.md#day3) |

1 日の時間割は `COURSE_SYLLABUS.md` に 1 つだけあります。このノートブックはその一部で、
自分の時間割を持ちません。時刻が分からなくなったら、まずそこを見てください。

The day's timetable exists once, in `COURSE_SYLLABUS.md`. This notebook is one part of it and
does not keep its own schedule; look there first when you are unsure of the time.


---

## Part 0　準備 / Preparation

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd().resolve()
while not ((COURSE_ROOT / "assets").is_dir() and (COURSE_ROOT / "notebooks").is_dir()):
    if COURSE_ROOT.parent == COURSE_ROOT:
        raise FileNotFoundError(
            "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
            "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
        )
    COURSE_ROOT = COURSE_ROOT.parent

ASSETS = COURSE_ROOT / "assets"
sys.path.insert(0, str(ASSETS / "helpers"))
import aivision

print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("PIL", "pillow"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
FONT_FAMILY = aivision.configure_japanese_font()
aivision.set_plot_style()

if FONT_FAMILY is None:
    print("日本語フォントが見つかりません。図のタイトルは英字になります。")
    print("No Japanese font found; figure titles fall back to English.")
else:
    print("図の日本語フォント / figure font:", FONT_FAMILY)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

def show(images, titles=None, figsize=(14, 5)):
    """画像を並べて表示する / display images side by side."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()


def detections_to_frame(result):
    """検出結果を pandas の表にする / convert detections into a DataFrame.

    ``result.boxes.data`` の 6 列は
    (x1, y1, x2, y2, confidence, class_id) です。
    """
    columns = ["class_id", "name", "confidence", "x1", "y1", "x2", "y2"]
    boxes = result.boxes
    if len(boxes) == 0:
        return pd.DataFrame(columns=columns)

    data = boxes.data.cpu().numpy()
    return pd.DataFrame({
        "class_id": data[:, 5].astype(int),
        "name": [result.names[int(class_id)] for class_id in data[:, 5]],
        "confidence": data[:, 4].round(4),
        "x1": data[:, 0].round(2),
        "y1": data[:, 1].round(2),
        "x2": data[:, 2].round(2),
        "y2": data[:, 3].round(2),
    })

#### YOLO とは
YOLO は **物体検出** のための AI モデルです。名前は
**You Only Look Once**（一度だけ見る）の頭文字です。

従来の方法では、画像を何度も走査して「ここに物体があるか？」を繰り返し調べました。
YOLO は **1 回の計算** で、画像全体の物体の位置と種類をまとめて予測します。
そのため **高速** で、製造ラインのような連続処理に向いています。

```text
入力：画像 1 枚
出力：複数の「四角 + 種類 + 確からしさ」
```

重要なのは、YOLO は「何かある」とは言うが、
**「何があるか」は学習した種類の中からしか答えられない** という点です。

#### What YOLO is
YOLO is an AI model for **object detection**. The name means
**You Only Look Once**.

Older methods scanned an image repeatedly asking "is there an object here?". YOLO predicts all
objects in the image in a **single pass**, which makes it **fast** and therefore suitable for
continuous production lines.

```text
input : one image
output: several "box + class + confidence" results
```

Crucially, YOLO can only answer **with the classes it was trained on**.

---

## Part 1　バウンディングボックス / Bounding boxes

### 四角を画像の上で読む / reading a box on the image

![](../../assets/diagrams/teach_bbox_coords.png)

図：同じ 1 つの欠陥の四角を、画素の数（左）と画像サイズで割った値（右）で表したもの。座標は計算して図に描いています。

Figure: one defect box written as pixel counts (left) and as fractions of the image size (right). The coordinates are computed and drawn, not sketched.

**気づいてほしいこと**

左の 4 つの数（x1, y1, x2, y2）と右の 4 つの数（cx, cy, w, h）は**同じ箱**です。0〜1 の値は「画像のどこか」を表すので、画像を縮小しても同じ意味になります。

**What to notice**

The four pixel numbers (x1, y1, x2, y2) and the four fractions (cx, cy, w, h) describe **the same box**. The 0–1 values say "where in the image", so they keep their meaning when the image is resized.

**考えてみてください / Try to answer**

- 右の cx=0.450 から、この箱の中心は画像の横幅の何割の位置にあると言えますか。
- From cx = 0.450, where is the centre of the box as a fraction of the image width?


#### 四角で囲む、4 つの数値
バウンディングボックス（以下「ボックス」）は、欠陥を囲む四角です。
指定方法は 2 通りあります。

| 方式 | 4 つの数値 | 使う場所 |
|---|---|---|
| 2 点方式 | `x1, y1, x2, y2`（左上と右下） | OpenCV、表示 |
| 中心方式 | `xc, yc, w, h`（中心と幅・高さ） | YOLO の正解データ |

YOLO の正解データは **中心方式** で、しかも
**画像の幅・高さで割った 0〜1 の値**（正規化）で書きます。
画像サイズが変わっても同じ値で使えるからです。

#### A rectangle is four numbers
A bounding box marks the defect with a rectangle, expressed in one of two ways:

| Style | Four numbers | Where it is used |
|---|---|---|
| corner style | `x1, y1, x2, y2` (top-left, bottom-right) | OpenCV, drawing |
| centre style | `xc, yc, w, h` (centre, width, height) | YOLO ground truth |

YOLO ground truth uses the **centre style**, and the numbers are **normalised by the image
width and height** into the range 0-1, so they stay valid if the image is resized.

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
show([Image.open(ASSETS / "diagrams" / "diagram_yolo_annotation.png")],
     ["YOLO annotation: normalised centre, width, height"], figsize=(9, 7))

#### 座標変換を自分で計算する
ピクセル座標 `(x1, y1, x2, y2)` から YOLO 形式への変換は 4 行です。

```text
xc = (x1 + x2) / 2 / 画像の幅
yc = (y1 + y2) / 2 / 画像の高さ
w  = (x2 - x1)     / 画像の幅
h  = (y2 - y1)     / 画像の高さ
```

割るのを忘れると、値が 1 を超えます。**0〜1 の範囲かどうかを必ず確認** してください。

#### Do the conversion yourself
Converting pixel corners to YOLO format takes four lines:

```text
xc = (x1 + x2) / 2 / image_width
yc = (y1 + y2) / 2 / image_height
w  = (x2 - x1)     / image_width
h  = (y2 - y1)     / image_height
```

Forgetting the division gives values above 1. **Always check the 0-1 range.**

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
image_width, image_height = 240, 240
x1, y1, x2, y2 = 96, 72, 168, 132

xc = (x1 + x2) / 2 / image_width
yc = (y1 + y2) / 2 / image_height
w = (x2 - x1) / image_width
h = (y2 - y1) / image_height

print(f"center x : {xc:.3f}")
print(f"center y : {yc:.3f}")
print(f"width    : {w:.3f}")
print(f"height   : {h:.3f}")
print()
print("0〜1 の範囲か / inside 0..1 ?", all(0 <= v <= 1 for v in (xc, yc, w, h)))
print("ラベルファイルの 1 行 / one label line:")
print(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

### 実行結果の例 / Expected output

次のような結果になります（数値は環境によって多少変わります）。

```text
center x : 0.550
center y : 0.425
width    : 0.300
height   : 0.250

0〜1 の範囲か / inside 0..1 ? True
ラベルファイルの 1 行 / one label line:
0 0.550000 0.425000 0.300000 0.250000
```

You should see something like the following (exact numbers vary slightly between machines).

```text
center x : 0.550
center y : 0.425
width    : 0.300
height   : 0.250

inside 0..1 ? True
0 0.550000 0.425000 0.300000 0.250000
```

---

## Part 2　クラスと確からしさ / Classes and confidence

#### クラスラベル
**クラス** は「種類」のことです。YOLO は、学習した種類のどれかに分類します。

一般的な事前学習モデル（COCO データセット）は 80 種類を学習しています。
例：`person`, `car`, `bus`, `dog`, `bottle` ……

ここに **`black_spot`（黒点）や `scratch`（傷）はありません**。
だから PVC の欠陥は検出できません。

#### Class label
A **class** is a category. YOLO assigns each detection to one of its trained classes.

The common pretrained model (COCO dataset) knows 80 classes, for example
`person`, `car`, `bus`, `dog`, `bottle` ...

**`black_spot` and `scratch` are not among them**, which is why it cannot detect PVC defects.

#### 確からしさ（confidence）
**confidence** は 0〜1 の数値で、モデル自身の「自信」です。

| 値 | モデルの気持ち（やさしく） |
|---|---|
| 0.95 | かなり自信がある |
| 0.50 | 半々くらい |
| 0.10 | ほとんど自信がない |

> これは **確率ではありません**。「この四角の中に、
> 学習した種類の物体が、私の知っている見た目で入っている度合い」です。
> **自信がある ≠ 正しい**。ここが重要です。

#### Confidence
**Confidence** is a number between 0 and 1: the model's own certainty.

| Value | What it means |
|---|---|
| 0.95 | very confident |
| 0.50 | about even |
| 0.10 | barely confident |

It is **not a probability**. It expresses "how much this box looks like the class I learned".
**High confidence does not mean correct** — that matters a lot.

---

## Part 3　IoU（重なりの度合い）/ Intersection over Union

### 交差・合計・IoU / intersection, union and IoU

![](../../assets/diagrams/teach_iou_rects.png)

図：A（正解）と B（予測）の四角を実際に重ね、交差と合計を計算したもの。

Figure: truth box A and predicted box B actually overlapped, with intersection and union computed.

**気づいてほしいこと**

IoU は「重なりの度合い」を 1 つの数にしたものです。0 に近いほど重なりが少なく、1 に近いほどぴったり一致しています。**面積の比** なので、単位はありません。

**What to notice**

IoU turns overlap into one number. Near 0 means little overlap; near 1 means a tight fit. It is a **ratio of areas**, so it has no unit.

**考えてみてください / Try to answer**

- B を右に 40 px ずらすと、IoU は上がりますか、下がりますか。
- If B moves 40 px to the right, does IoU go up or down?


#### IoU とは「2 つの四角の重なり具合」
**IoU** は、予測したボックスと正解のボックスが
**どれくらい重なっているか** を 0〜1 で表す数値です。

```text
IoU = 重なっている面積 ÷ 合計の面積
```

- `IoU = 1.0` … 完全に一致
- `IoU = 0.5` … まあまあ合っている
- `IoU = 0.0` … まったく重なっていない

一般に `IoU >= 0.5` を「正しく検出できた」とみなすことが多いです。
ただし、これは **慣習** であり、用途によって変わります。

#### IoU measures how much two boxes overlap
**IoU** expresses how much a predicted box overlaps the true box, between 0 and 1.

```text
IoU = area of overlap / area of union
```

- `IoU = 1.0` identical
- `IoU = 0.5` roughly matching
- `IoU = 0.0` no overlap

A detection is often counted as correct when `IoU >= 0.5`, though that is a **convention** that
depends on the application.

In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
def iou(box_a, box_b):
    """2 つのボックスの IoU を計算する / compute IoU of two (x1, y1, x2, y2) boxes."""
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    # 重なっている部分 / the intersection rectangle
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    intersection = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0


truth = (100, 100, 200, 200)          # 100 x 100 = 10,000 px2
cases = [
    ("完全一致 / identical      ", (100, 100, 200, 200)),
    ("少しずれる / shifted      ", (110, 110, 210, 210)),
    ("半分ずれる / half overlap ", (150, 100, 250, 200)),
    ("まったく別 / disjoint     ", (300, 300, 400, 400)),
]

print("正解 / truth :", truth)
print()
for label, box in cases:
    print(f"{label}  {box}  ->  IoU = {iou(truth, box):.3f}")

### 実行結果の例 / Expected output

次のような結果になります（数値は環境によって多少変わります）。

```text
正解 / truth : (100, 100, 200, 200)

完全一致 / identical        (100, 100, 200, 200)  ->  IoU = 1.000
少しずれる / shifted        (110, 110, 210, 210)  ->  IoU = 0.681
半分ずれる / half overlap   (150, 100, 250, 200)  ->  IoU = 0.333
まったく別 / disjoint       (300, 300, 400, 400)  ->  IoU = 0.000
```

You should see something like the following (exact numbers vary slightly between machines).

```text
truth : (100, 100, 200, 200)

identical        ->  IoU = 1.000
shifted          ->  IoU = 0.681
half overlap     ->  IoU = 0.333
disjoint         ->  IoU = 0.000
```

#### 手計算で確認する
「少しずれる」場合を手で計算してみましょう。

- 正解：`(100,100)` 〜 `(200,200)` → 面積 **10,000**
- 予測：`(110,110)` 〜 `(210,210)` → 面積 **10,000**
- 重なり：`(110,110)` 〜 `(200,200)` → 90 × 90 = **8,100**
- 合計：10,000 + 10,000 − 8,100 = **11,900**
#### Verify by hand
- IoU = 8,100 ÷ 11,900 ≒ **0.681**

Let us compute the "shifted" case by hand.

- truth: `(100,100)` to `(200,200)` -> area **10,000**
- prediction: `(110,110)` to `(210,210)` -> area **10,000**
- overlap: `(110,110)` to `(200,200)` -> 90 x 90 = **8,100**
- union: 10,000 + 10,000 - 8,100 = **11,900**
- IoU = 8,100 / 11,900 = **0.681**

---

#### 四角を動かして、交差・合計・IoU を見る
#### Move a box and watch the intersection, the union and IoU

`offset_x` と `offset_y` だけを動かします。**予測の四角** が動きます。

Only `offset_x` and `offset_y` change — the **predicted** box moves.

| 予想 / Predict | 見るもの / What to look for |
|---|---|
| ずれを 0 にする / offset 0 | IoU はいくつ？ / what is IoU? |
| ずれを大きくする / increase the offset | 交差と合計はどうなる？ / what happens to intersection and union? |
| まったく重ならない / no overlap | IoU はいくつ？ / what is IoU? |

ウィジェットが無くても、下のパラメータセルで同じ実験ができます。
The parameter cell below runs the same experiment without a widget.


In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
import matplotlib.pyplot as plt

truth_box = (60, 60, 110, 90)     # (x, y, 幅, 高さ) / (x, y, width, height)
offset_x = 40                     # ← 動かす値 / change this
offset_y = 20                     # ← 動かす値 / change this
predicted_box = (truth_box[0] + offset_x, truth_box[1] + offset_y, truth_box[2], truth_box[3])


def intersection_and_union(box_a, box_b):
    left = max(box_a[0], box_b[0])
    top = max(box_a[1], box_b[1])
    right = min(box_a[0] + box_a[2], box_b[0] + box_b[2])
    bottom = min(box_a[1] + box_a[3], box_b[1] + box_b[3])
    intersection = max(0, right - left) * max(0, bottom - top)
    union = box_a[2] * box_a[3] + box_b[2] * box_b[3] - intersection
    return intersection, union


intersection, union = intersection_and_union(truth_box, predicted_box)
print(f"交差 / intersection = {intersection} px²")
print(f"合計 / union        = {union} px²")
print(f"IoU                 = {iou(truth_box, predicted_box):.3f}")

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.set_xlim(0, 300); ax.set_ylim(260, 0); ax.grid(alpha=0.2)
ax.add_patch(plt.Rectangle(truth_box[:2], truth_box[2], truth_box[3], fill=False,
                           edgecolor="#1f6feb", linewidth=2))
ax.add_patch(plt.Rectangle(predicted_box[:2], predicted_box[2], predicted_box[3], fill=False,
                           edgecolor="#9a6700", linewidth=2, linestyle="--"))
ax.set_title(f"IoU = {iou(truth_box, predicted_box):.3f}", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
if not aivision.widgets_available():
    aivision.explain_widget_requirement()
else:
    import ipywidgets as widgets
    import matplotlib.pyplot as plt
    from IPython.display import clear_output, display

    output = widgets.Output()

    def draw(offset_x, offset_y):
        box_a = (60, 60, 110, 90)
        box_b = (box_a[0] + offset_x, box_a[1] + offset_y, box_a[2], box_a[3])
        left, top = max(box_a[0], box_b[0]), max(box_a[1], box_b[1])
        right = min(box_a[0] + box_a[2], box_b[0] + box_b[2])
        bottom = min(box_a[1] + box_a[3], box_b[1] + box_b[3])
        intersection = max(0, right - left) * max(0, bottom - top)
        union = box_a[2] * box_a[3] + box_b[2] * box_b[3] - intersection
        fig, ax = plt.subplots(figsize=(5.2, 5.2))
        ax.set_xlim(0, 300); ax.set_ylim(260, 0); ax.grid(alpha=0.2)
        ax.add_patch(plt.Rectangle(box_a[:2], box_a[2], box_a[3], fill=False,
                                   edgecolor="#1f6feb", linewidth=2))
        ax.add_patch(plt.Rectangle(box_b[:2], box_b[2], box_b[3], fill=False,
                                   edgecolor="#9a6700", linewidth=2, linestyle="--"))
        ax.set_title(f"intersection={intersection}  union={union}  "
                     f"IoU={iou(box_a, box_b):.3f}", fontsize=10)
        with output:
            clear_output(wait=True)
            plt.show()
            plt.close(fig)

    widgets.interact(draw,
                     offset_x=widgets.IntSlider(value=40, min=-100, max=140, step=5,
                                                description="offset x"),
                     offset_y=widgets.IntSlider(value=20, min=-100, max=140, step=5,
                                                description="offset y"))
    display(output)

## Part 4　事前学習モデルと重みファイル / Pretrained models and weight files

#### 「学習済み」を持ってくる
ゼロから学習すると、数万枚の画像と数日〜数週間かかります。
そこで、**すでに大量の画像で学習済みのモデル** を使います。これを
**事前学習モデル（pretrained model）** と呼びます。

**重みファイル（weight file）** は、その学習結果が入ったファイルです。
拡張子は `.pt`（PyTorch）や `.onnx` です。

| 用語 | 意味 |
|---|---|
| 事前学習モデル | 他のデータで学習済みのモデル |
| 重み（weights） | 学習で決まった数値の集まり |
| 推論（inference） | 学習済みモデルで新しい画像を判定すること |
| ファインチューニング | 事前学習モデルを、自分のデータで追加学習すること |

第 4 日で行うのは **ファインチューニング** です。
COCO で学んだ「物の形を見る力」を借りて、PVC の欠陥を覚えさせます。

#### Starting from something already trained
Training from scratch needs tens of thousands of images and days or weeks.
Instead we start from a model **already trained on a large dataset**: a
**pretrained model**.

A **weight file** holds the result of that training (`.pt` for PyTorch, `.onnx` for ONNX).

| Term | Meaning |
|---|---|
| pretrained model | a model already trained on other data |
| weights | the numbers determined by training |
| inference | applying a trained model to new images |
| fine-tuning | further training of a pretrained model on your own data |

Day 4 performs **fine-tuning**: borrow the "shape understanding" learned on COCO and teach it
PVC defects.

#### なぜ COCO 事前学習モデルは PVC 欠陥を検出できないか
理由は 4 つあります。

1. **クラスが無い** — COCO の 80 種類に `black_spot` は無い。
   モデルは「黒い点」を何かに分類しようとするが、正解が無い。
2. **見た目が違う** — COCO の物体は「人」「車」など、背景からはっきり分離した大きな物体。
   シート上の 0.3 mm の欠陥は、学習した見た目とまったく違う。
3. **大きさが違う** — 学習した物体は画像の中で大きく写っている。
   微小な欠陥は、モデルが想定していない大きさ。
4. **正解データが無い** — そもそも「これは欠陥」というラベルで学習していない。

だから第 4 日で、**自分のデータで学習（ファインチューニング）** します。
第 3 日は「道具の使い方」を覚える日だと考えてください。

#### Why a COCO-pretrained model cannot detect PVC defects
Four reasons:

1. **no such class** — `black_spot` is not among the 80 COCO classes
2. **different appearance** — COCO objects are large, clearly separated things like people or cars;
   a 0.3 mm mark on a sheet looks nothing like them
3. **different scale** — the learned objects are large in frame, while these defects are tiny
4. **no supervision** — the model was never trained with "this is a defect" labels

That is why Day 4 **fine-tunes on your own data**. Day 3 is about learning to use the tool.

---

## 演習 / Exercises

#### 演習 1（基本）：YOLO 形式に変換する
画像サイズが 640 × 480、ボックスが `x1=320, y1=200, x2=400, y2=260` のとき、
YOLO 形式の 4 つの値を計算してください。

#### Exercise 1 (basic): convert to YOLO format
For a 640x480 image and a box `x1=320, y1=200, x2=400, y2=260`, compute the four
YOLO values.
Done when all four values are between 0 and 1 and printed with 6 decimals.

In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 1 / Exercise 1
width, height = 640, 480
x1, y1, x2, y2 = 320, 200, 400, 260

xc = 0.0   # <- 計算 / compute
yc = 0.0
w = 0.0
h = 0.0

print(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

#### 演習 2（中級）：IoU を手計算して確かめる
正解 `(50, 50, 150, 150)`、予測 `(100, 50, 200, 150)` の IoU を
**紙で計算** してから、下のセルで確認してください。

#### Exercise 2 (intermediate): a hand calculation of IoU
Compute the IoU of truth `(50, 50, 150, 150)` and prediction `(100, 50, 200, 150)`
**on paper**, then check it with the cell.
Done when your paper result matches `iou(...)` (it is 0.333).

In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 2 / Exercise 2
print("紙の計算 / on paper:", "")
print("コードの値 / code   :", round(iou((50, 50, 150, 150), (100, 50, 200, 150)), 3))

#### 演習 3（発展）（発展・任意）：第4日で何を学ぶ必要があるかを整理する
事前学習モデルが PVC 欠陥を検出できない 4 つの理由を踏まえ、
第 4 日で用意すべきものを整理してください。

| 項目 | あなたの答え |
|---|---|
| クラス名 | |
| 1 枚あたりの欠陥数（目安） | |
| 必要な画像枚数（学習用） | |
| 正解データの形式 | |

#### Exercise 3 (challenge) (challenge, optional) plan what Day 4 must teach the model
Given the four reasons, plan what Day 4 must prepare: class names, defects per image,
number of training images, and the ground-truth format.
Done when the table is filled and the label format states the YOLO centre format.

In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 3 / Exercise 3
plan = {
    "classes": [],
    "defects_per_image": "",
    "training_images": "",
    "label_format": "",
}
for key, value in plan.items():
    print(f"{key:<20} {value if value else '(未記入 / not written)'}")

---

#### 理解度チェック / Understanding questions

1. YOLO の出力は何ですか。3 つの要素を挙げてください。
2. confidence が 0.9 の検出は、必ず正しいですか。理由も書いてください。
3. IoU が 0.3 のとき、予測はどれくらい合っていますか。
4. 事前学習モデルが PVC 欠陥を検出できない理由を 2 つ挙げてください。
5. 「推論」と「学習」の違いを、時間の観点から説明してください。

1. What does YOLO output? Name three components.
2. Is a detection with confidence 0.9 always correct? Explain.
3. What does an IoU of 0.3 tell you about a prediction?
4. Give two reasons a pretrained model cannot detect PVC defects.
5. Explain the difference between inference and training in terms of time.
Model answers: `docs/instructor/day03_instructor_guide.md`.

---

## 困ったときは / Troubleshooting

エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| YOLO 形式の値が 1 を超える | YOLO values exceed 1 | 画像の幅・高さで割り忘れています。`/ image_width` を確認してください。 | You forgot to divide by the image size. |
| IoU がマイナスになる | IoU is negative | 重なりが無いときは 0 になります。`max(0.0, ...)` を忘れていないか確認してください。 | It should be 0 when there is no overlap; check `max(0.0, ...)`. |
| `confidence` と `確率` を混同してしまう | Confusing confidence with probability | confidence はモデルの自己評価です。正解率ではありません。 | Confidence is the model's self-assessment, not an accuracy. |
| 数値がノートの例と少し違う | Numbers differ slightly | 丸めの桁数が違うだけのことが多いです。`round()` の桁を確認してください。 | Usually just a rounding difference; check the digits in `round()`. |

## 完了チェックリスト / Completion checklist

**必須の項目だけ** を並べています。**演習 3（発展）は任意** で、未実施でも完了です。時間が余ったときだけ取り組んでください。

Only the **required** items are listed. **Exercise 3 (challenge) is optional**; leaving it undone still counts as complete. Do it only if you have spare time.

すべてにチェックが付いたら、このノートブックは完了です。

This notebook is complete when every box is checked.

- [ ] Part 1〜Part 4 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-4 and ran every code cell
- [ ] YOLO 形式の座標を自分で計算できた
      - [ ] Computed YOLO coordinates yourself
- [ ] IoU を手計算し、コードと一致した
      - [ ] Hand-computed IoU and matched the code
- [ ] 演習 1〜2（必須）を実施した
      - [ ] Completed exercises 1-2 (required)
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Ultralytics ドキュメント](https://docs.ultralytics.com/)
- [Ultralytics: 物体検出タスク](https://docs.ultralytics.com/tasks/detect/)
- [COCO データセットのクラス一覧](https://cocodataset.org/#explore)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

COCO の 80 クラスの一覧を見て、製造検査に使えそうなクラスがあるか探してください。

Look at the 80 COCO classes and see whether any of them could be useful for inspection.